In [156]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [270]:
df_new = pd.read_csv('OctNov_CarData.csv')
df_old = pd.read_csv('total_call_data.csv')
df_new.shape

C:\Users\Owner\AppData\Local\Temp\ipykernel_37484\2688094094.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_old = pd.read_csv('total_call_data.csv')


(331992, 9)

In [272]:
# Split old data set into before and after Oct 15
# Convert timestamps
df_old['Activity Start Timestamp'] = pd.to_datetime(df_old['Activity Start Timestamp'], format='mixed')
df_new['Activity Start Timestamp'] = pd.to_datetime(df_new['Activity Start Timestamp'], format='mixed')

# Get the first timestamp for each Contact Session ID in df_new
first_timestamps = df_new.groupby('Contact Session ID')['Activity Start Timestamp'].min()

# Create a mapping of Contact Session ID to whether it should go to old dataset
cutoff_date = pd.to_datetime('2025-10-15 23:59:59')
session_to_old = first_timestamps <= cutoff_date

# Split df_new based on the first timestamp of each session
df_new_before = df_new[df_new['Contact Session ID'].map(session_to_old).fillna(False)]
df_new = df_new[~df_new['Contact Session ID'].map(session_to_old).fillna(True)]

# Sort df_new by Contact Session ID and Activity Start Timestamp
df_new = df_new.sort_values(['Contact Session ID', 'Activity Start Timestamp']).reset_index(drop=True)

# Concatenate the sessions that started before cutoff to df_old
# df_old = pd.concat([df_old, df_new_before])

# Sort df_old by Contact Session ID and Activity Start Timestamp
df_old = df_old.sort_values(['Contact Session ID', 'Activity Start Timestamp']).reset_index(drop=True)


In [273]:
# First Step: Check if proportion of senior menu queues is the same in both datasets
# Filter out all queues that contain transfer or intake outdial

ids_senior_new = df_new[df_new['Activity Name'].str.contains('SuburbsOrCityMenu', na=False)]['Contact Session ID'].unique()
ids_senior_old = df_old[df_old['Activity Name'].str.contains('SuburbsOrCityMenu', na=False)]['Contact Session ID'].unique()


prop_senior_new = len(ids_senior_new) / df_new['Contact Session ID'].nunique()
prop_senior_old = len(ids_senior_old) / df_old['Contact Session ID'].nunique()

print(f"prop new: {prop_senior_new * 100:.4f} %, prop old: {prop_senior_old* 100:.4f} %")
print(f"Total CitySenior new IDs: {len(ids_senior_new)}, total new IDS {df_new['Contact Session ID'].nunique()},  CitySenior old IDs: {len(ids_senior_old)}, total old IDs {df_old['Contact Session ID'].nunique()}")

# Making dataframe with just seniors, Idt this is necessary though
senior_old_df = df_old.loc[df_old['Contact Session ID'].isin(ids_senior_old)]
senior_new_df = df_new.loc[df_new['Contact Session ID'].isin(ids_senior_new)]

# Adding column in dataframe that signifies senior or not seniors
df_new['Senior'] = df_new['Contact Session ID'].isin(ids_senior_new)
df_old['Senior'] = df_old['Contact Session ID'].isin(ids_senior_old)

prop new: 7.1297 %, prop old: 11.3171 %
Total CitySenior new IDs: 1359, total new IDS 19061,  CitySenior old IDs: 27228, total old IDs 240592


In [274]:
# Create 2 new dataframes
# Columns: Contact Session ID, Time of SeniorMenu Traversal, Activity Start Timestamp

def senior_menu_time(df):
    # Filter for rows containing the menu items
    seniors = df[df['Activity Name'] == 'SeniorsMenu'][['Contact Session ID', 'Activity Start Timestamp']]
    suburbs = df[df['Activity Name'] == 'SuburbsOrCityMenu'][['Contact Session ID', 'Activity Start Timestamp']]
    
    # Get the first occurrence of each menu for each Contact Session ID, looks simply at fast the traversal is
    seniors_first = seniors.groupby('Contact Session ID')['Activity Start Timestamp'].min().rename('SeniorsMenu_time')
    suburbs_first = suburbs.groupby('Contact Session ID')['Activity Start Timestamp'].min().rename('SuburbsOrCityMenu_first_time')
    # Also considers confusion of people retraversing through menus
    suburbs_last = suburbs.groupby('Contact Session ID')['Activity Start Timestamp'].max().rename('SuburbsOrCityMenu_last_time')

    
    # Merge the timestamps
    times = pd.merge(seniors_first, suburbs_first, left_index=True, right_index=True, how='inner')
    time2 = pd.merge(seniors_first, suburbs_last, left_index=True, right_index=True, how='inner')
    
    # Calculate the difference in seconds
    times['time_diff_first'] = (times['SuburbsOrCityMenu_first_time'] - times['SeniorsMenu_time']).dt.total_seconds()
    time2['time_diff_last'] = (time2['SuburbsOrCityMenu_last_time'] - time2['SeniorsMenu_time']).dt.total_seconds()
    
    # Merge both time dataframes
    combined = pd.merge(times, time2[['time_diff_last']], left_index=True, right_index=True, how='outer')
    
    # Get first Activity Start Timestamp for each Contact Session ID
    first_activity = df.groupby('Contact Session ID')['Activity Start Timestamp'].min()
    
    # Merge with combined times
    dataframe = pd.merge(combined, first_activity.rename('Activity Start Timestamp'), left_index=True, right_index=True, how='left')
    dataframe = dataframe.reset_index()
    
    return dataframe


old_times = senior_menu_time(senior_old_df)
new_times = senior_menu_time(senior_new_df)


In [275]:
# Create new dataframe, very simple
# 3 columns, Contact Session ID, Activity Start TimeStamp, Senior
def senior_prop(df):
    # Get the first activity timestamp for each Contact Session ID
    first_activity = df.groupby('Contact Session ID')['Activity Start Timestamp'].min()
    
    # Get the Senior value for each Contact Session ID (assuming it's the same for all rows with same ID)
    senior_value = df.groupby('Contact Session ID')['Senior'].first()
    
    # Combine into a dataframe
    result = pd.DataFrame({
        'Contact Session ID': first_activity.index,
        'Activity Start Timestamp': first_activity.values,
        'Senior': senior_value.values
    })
    
    return result

new_senior_prop = senior_prop(df_new)
old_senior_prop = senior_prop(df_old)

In [279]:
old_times['time_diff_last'].mean()


np.float64(24.364183928309092)

In [280]:
new_times['time_diff_last'].mean()

np.float64(15.708609271523178)

In [346]:
old_times['time_diff_first'].mean()


np.float64(23.744344057587778)

In [345]:
new_times['time_diff_first'].mean()


np.float64(7.178807947019868)

In [ ]:
# Time of traversal of senior menu between new and old
old_times.to_excel('Senior_Menu_Time_Old.xlsx')
new_times.to_excel('Senior_Menu_Time_New.xlsx')
# Data for proportion of seniors who are callers, gonna use to see if November is a particularly low senior call month
old_senior_prop.to_excel('Old_CAR_Data.xlsx')
new_senior_prop.to_excel('New_CAR_Data.xlsx')

In [234]:
# Get next non-NA Activity Name after SuburbsOrCityMenu for df_new
def get_next_activity(group):
    # Find rows with SuburbsOrCityMenu
    mask = group['Activity Name'].str.contains('SuburbsOrCityMenu', na=False)
    if not mask.any():
        return None
    
    # Get the index of the first SuburbsOrCityMenu occurrence
    first_idx = group[mask].index[0]
    
    # Get all rows after this index
    after_rows = group.loc[group.index > first_idx]
    
    # Find the first non-NA Activity Name
    next_activities = after_rows['Activity Name'].dropna()
    if len(next_activities) > 0:
        return next_activities.iloc[0]
    return None


# Apply to each Contact Session ID
next_activities = df_new.groupby('Contact Session ID').apply(get_next_activity)
next_activities_old = df_old.groupby('Contact Session ID').apply(get_next_activity)
print("\nNext activities after SuburbsOrCityMenu (value counts):")
print(next_activities.value_counts())
print("\nNext activities after SuburbsOrCityMenu for old df (value counts):")
print(next_activities_old.value_counts())

C:\Users\Owner\AppData\Local\Temp\ipykernel_37484\3210543388.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  next_activities = df_new.groupby('Contact Session ID').apply(get_next_activity)



Next activities after SuburbsOrCityMenu (value counts):
ClosedQueueMenu                           746
GetLoggedInSubSeniorFamilyAgents           63
SeniorNotCookCoMenu                        63
TenantDeterrenceMenu                       60
GetLoggedInSubSeniorHomeownerAgents        51
GetLoggedInSubSeniorBenefitsAgents         50
GetLoggedInSubSeniorConsumerAgents         48
GetLoggedInConsumerAgents                  47
GetLoggedInSubSeniorADAPTAgents            40
GetLoggedInSubSeniorTenantAgents           38
GetLoggedInFamilyAgents                    36
GetLoggedInBenefitsAgents                  21
GetLoggedInHousingAgents                   19
SuburbsOrCityMenu                          14
GetLoggedInADAPTAgents                     12
GetLoggedInSubSeniorEmploymentAgents       11
GetLoggedInSubSeniorConsumerSPAgents        9
GetLoggedInSubSeniorBenefitsSPAgents        4
GetLoggedInSubSeniorFamilySPAgents          4
GetLoggedInSubSeniorADAPTSPAgents           4
GetLoggedInEmploymentAg

C:\Users\Owner\AppData\Local\Temp\ipykernel_37484\3210543388.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  next_activities_old = df_old.groupby('Contact Session ID').apply(get_next_activity)


In [328]:
ids_senior_new = df_new[(df_new['Activity Name'].str.contains('SeniorsMenu', na=False))]['Contact Session ID'].unique()
ids_senior_old = df_old[(df_old['Activity Name'].str.contains('SeniorsMenu', na=False)) | (df_old['Activity Name'].str.contains('SeniorsADAPTMenu', na=False))]['Contact Session ID'].unique()


prop_senior_new = len(ids_senior_new) / df_new['Contact Session ID'].nunique()
prop_senior_old = len(ids_senior_old) / df_old['Contact Session ID'].nunique()

print(f"prop new: {prop_senior_new * 100:.4f} %, prop old: {prop_senior_old* 100:.4f} %")
print(f"Total SubSenior new IDs: {len(ids_senior_new)}, total new IDS {df_new['Contact Session ID'].nunique()},  SubSenior old IDs: {len(ids_senior_old)}, total old IDs {df_old['Contact Session ID'].nunique()}")

prop new: 26.6775 %, prop old: 43.8714 %
Total SubSenior new IDs: 5085, total new IDS 19061,  SubSenior old IDs: 105551, total old IDs 240592


In [339]:
# check proportion of seniors reaching queue in old and new
bad_ids = df_new.loc[
    df_new['Activity Name'].str.contains('SuburbsOrCityMenu', na=False)]['Contact Session ID'].unique()

# Step 2: Select all IDs that are NOT in bad_ids
df_new_filt = df_new.loc[
    df_new['Contact Session ID'].isin(bad_ids)]

ids2 = df_new_filt.loc[(df_new_filt['Activity Name'].str.contains('ClosedQueueMenu', na=False)) | (df_new_filt['Activity Name'].str.contains('PreQueue', na=False))]['Contact Session ID'].unique()
df_new2 = df_new_filt[df_new_filt['Contact Session ID'].isin(ids2)]

df_new2.head()

# check proportion of seniors reaching queue in old and new
bad_ids = df_old.loc[
    df_old['Activity Name'].str.contains('SuburbsOrCityMenu', na=False)]['Contact Session ID'].unique()

# Step 2: Select all IDs that are NOT in bad_ids
df_old_filt = df_old.loc[
    df_old['Contact Session ID'].isin(bad_ids)]

ids2 = df_old_filt.loc[(df_old_filt['Activity Name'].str.contains('ClosedQueueMenu', na=False)) | (df_old_filt['Activity Name'].str.contains('PreQueue', na=False))]['Contact Session ID'].unique()
df_old2 = df_old_filt[df_old_filt['Contact Session ID'].isin(ids2)]


In [340]:
df_new2['Contact Session ID'].nunique() /df_new['Contact Session ID'].nunique()

0.06521168878862599

In [341]:
df_old2['Contact Session ID'].nunique() / df_old['Contact Session ID'].nunique()

0.07286609696082995

In [342]:
# check proportion of non seniors reaching queue in old and new
bad_ids = df_new.loc[
    df_new['Activity Name'].str.contains('SuburbsOrCityMenu', na=False)]['Contact Session ID'].unique()

# Step 2: Select all IDs that are NOT in bad_ids
df_new_filt = df_new.loc[
    ~df_new['Contact Session ID'].isin(bad_ids)]

ids2 = df_new_filt.loc[(df_new_filt['Activity Name'].str.contains('ClosedQueueMenu', na=False)) | (df_new_filt['Activity Name'].str.contains('PreQueue', na=False))]['Contact Session ID'].unique()
df_new2 = df_new_filt[df_new_filt['Contact Session ID'].isin(ids2)]

df_new2.head()

# check proportion of non seniors reaching queue in old and new
bad_ids = df_old.loc[
    df_old['Activity Name'].str.contains('SuburbsOrCityMenu', na=False)]['Contact Session ID'].unique()

# Step 2: Select all IDs that are NOT in bad_ids
df_old_filt = df_old.loc[
    ~df_old['Contact Session ID'].isin(bad_ids)]

ids2 = df_old_filt.loc[(df_old_filt['Activity Name'].str.contains('ClosedQueueMenu', na=False)) | (df_old_filt['Activity Name'].str.contains('PreQueue', na=False))]['Contact Session ID'].unique()
df_old2 = df_old_filt[df_old_filt['Contact Session ID'].isin(ids2)]


In [343]:
df_new2['Contact Session ID'].nunique() /df_new['Contact Session ID'].nunique()

0.19164786737317036

In [344]:
df_old2['Contact Session ID'].nunique() / df_old['Contact Session ID'].nunique()

0.1772128749085589